# 02_custom_auth_coordinator — Write your own authentication coordinator

An authentication coordinator turns a raw transport request into an execution `Context` *before* the machine runs. The whole contract is one async method, `process(request) -> Context | None`. We compose the shipped `AuthCoordinator` from three small parts — extractor, authenticator, assembler — and drive it with a stand-in request.

Companion guide: [Your own auth coordinator](../../docs/how-to/authoring-auth-coordinator.md).

In [ ]:
!pip install aoa-action-machine

In [ ]:
import asyncio
from dataclasses import dataclass, field
from typing import Any

from aoa.action_machine.auth import (

## The example

In [ ]:
    AuthCoordinator,
    Authenticator,
    BaseRole,
    ContextAssembler,
    CredentialExtractor,
)
from aoa.action_machine.context import Context, UserInfo

In [ ]:
# ── A role the caller may carry (type-as-capability, name ends in "Role") ────
class AdminRole(BaseRole):
    name = "admin"
    description = "Full administrative access."

In [ ]:
# ── A stand-in for a transport request (FastAPI Request, MCP ctx, ...) ───────
@dataclass
class FakeRequest:
    headers: dict[str, str] = field(default_factory=dict)
    path: str = "/"


# Pretend key store: api-key -> (user_id, roles)
_KEYS = {"secret-123": ("agent_7", (AdminRole,))}

In [ ]:
# ── 1. Extractor: pull credentials out of the protocol request ───────────────
class ApiKeyExtractor(CredentialExtractor):
    async def extract(self, request_data: Any) -> dict[str, Any]:
        key = request_data.headers.get("x-api-key")
        return {"api_key": key} if key else {}     # empty dict => no credentials

In [ ]:
# ── 2. Authenticator: verify credentials, resolve a UserInfo ─────────────────
class ApiKeyAuthenticator(Authenticator):
    async def authenticate(self, credentials: Any) -> UserInfo | None:
        record = _KEYS.get(credentials.get("api_key"))
        if record is None:
            return None                             # invalid => None, never raise
        user_id, roles = record
        return UserInfo(user_id=user_id, roles=roles)

In [ ]:
# ── 3. Assembler: project request metadata into RequestInfo kwargs ───────────
class HttpContextAssembler(ContextAssembler):
    async def assemble(self, request_data: Any) -> dict[str, Any]:
        return {
            "request_path": request_data.path,
            "trace_id": request_data.headers.get("x-trace-id"),
            "protocol": "http",
        }

## Run

Colab already runs an event loop, so call `await main()` at the top level instead of `asyncio.run(main())`.

In [ ]:
async def main() -> None:
    # Compose the three parts into the shipped orchestration.
    coordinator = AuthCoordinator(
        ApiKeyExtractor(), ApiKeyAuthenticator(), HttpContextAssembler(),
    )

    # Valid key -> populated Context.
    ok = await coordinator.process(FakeRequest(
        headers={"x-api-key": "secret-123", "x-trace-id": "abc-1"}, path="/orders",
    ))
    print("valid key   ->", ok and (ok.user.user_id, [r.name for r in ok.user.roles], ok.request.trace_id))

    # Bad key -> None (adapter would fall back to anonymous Context()).
    bad = await coordinator.process(FakeRequest(headers={"x-api-key": "nope"}, path="/orders"))
    print("invalid key ->", bad)

    # No credentials -> None as well (extractor returned empty dict).
    anon = await coordinator.process(FakeRequest(path="/orders"))
    print("no creds    ->", anon)

await main()